# Day I - A) First steps in FEM universe

- **<font color='green'>[RUN & OBSERVE]</font>** $\rightarrow$ the cell should be run directly without modification
- **<font color='orange'>[RUN & PLAY]</font>** $\rightarrow$ the cell can be run directly, but some parameters should be changed interactively
- **<font color='red'>[FILL & RUN]</font>**    $\rightarrow$ the cell should be filled before being run
- **<font color='magenta'>[FILL & PLAY]</font>** $\rightarrow$ the cell should be filled, and then some parameters should be changed interactively.


## Outlines:

> [1) Geometry](#1_geometry)
>> [a) Continuous domain](#1a_continuous_domain)
>>
>> [b) Discretized domain](#1b_discretized_domain)

|**<u>Objective</u>** : Discover the mesh, function space and variational formulations|
|---|
|*Indicative duration* : 1 hour|

<a id='1_geometry'></a>


---

## 1) Geometric domain

### 1.a) Continuous domain

<a id='1a_continuous_domain'></a>

We consider a domain $\Omega$; it may have an inclusion called $\mathrm{in}$.

|                | Square                                | Disk                                                       |
|----------------|---------------------------------------|------------------------------------------------------------|
|                | $\Omega = [0,1]^2$                    | $\Omega = \{x\in \mathbb R^2, \|\|x\|\| \leq 1\}$          |
| if `inclusion` | $\mathrm{in} = [0.25, 0.75]^2$        | $\mathrm{in} = \{ x \in \mathbb R^2, \|\|x\|\| \leq 0.5\}$ |
| else           | $\mathrm{in} = \emptyset$             | $\mathrm{in} = \emptyset$                                  |
|                | $\mathrm{out} = \Omega / \mathrm{in}$ | $\mathrm{out} = \Omega / \mathrm{in}$                      |

**<font color='orange'>[RUN & PLAY]</font>**

In [ ]:
# Define the geometric domain

# Try out different geometries
from utils.myGeometries import square, disk
#-------------------------------------------------------------
geometry = square     # possible choices : square, disk
inclusion = False     # possible choices : True, False
#-------------------------------------------------------------

Omega = geometry(draw = True, inclusion = inclusion)

---
### 1.b) Discretized domain

<a id='1b_discretized_domain'></a>


We partition $\Omega$ into elements to create a mesh.
The side length of an element is usually called $h$. So, the discrtized domain is usually called $\Omega_h$. In 2D, usual elements can be *triangles* or *quadrangles*. We can have both element types in the same mesh.

Generating the mesh is usually a complex task done by third-party software, such as [GMSH](https://gmsh.info/), [Netgen](https://github.com/ngsolve/netgen), or [Salome](https://www.salome-platform.org/?page_id=374). 

In this tutorial, we use [NGSolve](https://ngsolve.org/) finite element toolbox, which include Netgen. We can play with different parameters.

**<font color='orange'>[RUN & PLAY]</font>**

In [ ]:
# Generate the discrete mesh with tunable parameters

# Try out different parameters with the disk and square and see the effect!
#-------------------------------------------------------------
maxh = 0.2              # maximum element size (>0.01)
quad_dominated = False  # try to mesh mainly with quadrangles
curve_order = 1         # curve the mesh elements
#-------------------------------------------------------------


from utils.myGeometries import Mesh

Omega_h = Mesh(Omega, 
               maxh = maxh, 
               quad_dominated = quad_dominated       
               ).Curve(curve_order)

from ngsolve.webgui import Draw
print(f"Mesh generated with {Omega_h.nv} vertices, {Omega_h.nedge} edges, {Omega_h.ne} elements")
Draw(Omega_h)

---
## 2) Function definition

### 2.a) Continuous function

We consider the following analytical function that is continuous on $\Omega$:
$$ u_{H^1\text{ref}} : \left \{ \begin{array}{rcl}
\Omega \subset \mathbb R^2 & \rightarrow & \mathbb R \\
(x , y) &\mapsto & (x-0.5)^2 + (y-0.5)^2
\end{array} \right.$$

|**<font color='green'>[RUN & OBSERVE]</font>**|
|---|

In [ ]:
 # Define the analytic function, called "CoefficientFunction" in NGSolve

from ngsolve import x, y
uH1_ref = (x-0.5)**2 + (y-0.5)**2   

print(type(uH1_ref))

In [ ]:
# We can look at the expression tree. This function is not attached to any discretization yet.

print(uH1_ref)

In [ ]:
# We can also plot the function. To do so, we need a discretized domain, so that the
# CoefficientFunction can be evaluated from the expression tree within each element.

from ngsolve.webgui import Draw
print("Continuous function uH1_ref")
Draw(uH1_ref,            # our analytic function
     Omega_h,       # the mesh
     order = 7,     # define the polynomial order of the visualization.
     settings = { "Objects" : { "Wireframe" : False },  # to have a clean view
                 "deformation" :  1}
     )

### 2.b) Discontinuous function

Now, we consider the restriction of the function to the inclusion (if considered in the geometry):
$$ u_{L^2\text{ref}} : \left \{ \begin{array}{rcl}
\Omega = [0,1]^2 & \rightarrow & \mathbb R \\
(x,y) &\mapsto & \left\{
\begin{array}{ll}
u_{H^1 \text{ref}}(x)  & \text{if~} x \in \mathrm{In}\\
0 & \text{otherwise}
\end{array} \right.
\end{array} \right.$$

|**<font color='green'>[RUN & OBSERVE]</font>**|
|---|

In [ ]:
# Define the analytical function according to its definition

uL2_ref = Omega_h.MaterialCF({"in" : uH1_ref}, default = 0)

# Plot the function
print("Discontinuous function uL2_ref")

from ngsolve.webgui import Draw
Draw(uL2_ref,            # our analytic function
     Omega_h,       # the mesh
     order = 7,     # define the polynomial order of the visualization.
     settings = { "Objects" : { "Wireframe" : False },  # to have a clean view
                 "deformation" :  1}
     )

Now we chose the reference function $u_{ref}$ for the rest of the notebook.

**<font color='orange'>[RUN & PLAY]</font>**

In [ ]:
# Reference function for the rest of the notebook

u_ref = uH1_ref   # choices: uH1_ref or uL2_ref

Now, imagine the reference function $u$ as a black-box, we don't have access to its expression. We can only evaluate $u at point coordinates, but it's long to evaluate.  So, our goal is to build an approximation of $u$ that is fast to compute.

This can be done by building an **interpolation** of $u$ on $\Omega_h$. The question is now:

|How to build the closest interpolation of the reference function on $\Omega_h$?|
|---|


---
### 2.b) Function spaces

To define an interpolation, we need first to specify the type of *degrees of freedom* (DoFs).
There is a lot of different possibilities (see [this website](https://defelement.org/) for an overview). Here we address only the usual ones.

#### 2.b.i) $L^2(\Omega)$ (element DoFs $\Rightarrow$ discontinuous accross elements)

Consider the function space

$$ L^2(\Omega) = \left \{ v : \Omega \rightarrow \mathbb R, \int_\Omega |v|^2 \leq \infty \right \} $$

Such functions can be **discontinuous** almost everywhere.
Therefore, a possible discretization is to build the interpolation **element**-wise, allowing a possible **discontinuity**  between elements. Inside each element, the function is still continuous.

Such interpolations are called $P_0$ (if they are constant per element), "discontinuous Galerkin" or  "discontinuous Lagrange" in general.

|**<font color='orange'>[RUN & PLAY]</font>**|
|---|

In [ ]:
# Define a function on a L2 space discretized on Omega_h

from ngsolve import L2, GridFunction

polynomial_orderL2 = 0                            # can be changed
fesL2 = L2(Omega_h, order = polynomial_orderL2)   # define the discretized function space
uL2 = GridFunction(fesL2)                         # define a discretized function
uL2.vec.data[-1] = 1                              # set a DoF to 1

print(uL2)                                        # we can see that now, uL2 is attached to fesL2
Draw(uL2, Omega_h, order = polynomial_orderL2 + 1, 
     settings = {"deformation" :  0.5})           # vizualize the discretized function

#### 2.b.ii) $H^1(\Omega)$ (nodal DoF $\Rightarrow \mathcal C^0$ across elements)

$$ H^1(\Omega) = \left \{ u \in L^2(\Omega), \nabla u \in L^2(\Omega) \right \} $$

Such functions are be continuous (almost) everywhere.
So, the discretized version $H^1(\Omega_h)$ corresponds to the space of continuous functions, usually defined **node**-wise. Since nodes are shared between neighbouring elements, **no discontinuity** is possible.

Such continuous interpolations are called $P_n(\Omega_h)$, with $n\geq 1$ the polynomial order, or "Lagrange" elements.

|**<font color='magenta'>[FILL & PLAY]</font>**|
|---|

In [ ]:
from ngsolve import H1

polynomial_orderH1= 1  # can be changed
fesH1 = H1(Omega_h, order = polynomial_orderH1)     # define the discretized function space
uH1 = GridFunction(fesH1)                           # define a discretized function
uH1.vec.data[-1] = 1                                # set a DoF to 1
Draw(uH1, order=polynomial_orderH1+1, 
     settings = {"deformation" :  0.5})             # vizualize the function

**Question** : does a `H1` function space with `order = 0` make sense?

|Now that the function spaces are defined, how to choose the best interpolation out of them? |
|---|

---
### 2.c) Least square formulation

A reasonable criteria to find the best possible approximation is to look for a function $u^*\in H$ (with $H=L^2(\Omega_h)$ or $H^1(\Omega_h)$ with arbitrary polynomial order, setting the regularity level we want), which minimizes the integral of the square error with the reference :
$$ u^* = \arg \min_{u\in H} J(u) = \frac{1}{2}\int_{x\in\Omega_h} ( u(x) -u_{H^1}(x) )^2 \; \mathrm{d} x$$

The integral can be approximated using a **finite number of functions calls** to $u$ ([quadrature methods](https://en.wikipedia.org/wiki/Gaussian_quadrature)), so this formulation can indeed be used even if $u$ is a black-box.

In what follows, we drop the $x$ dependency for conciseness :
- $u(x)\rightarrow u$ and $u_{H^1}(x) \rightarrow u_{H^1}$
- no "$ x\in $"  and "$\mathrm{d}x$" in the integrals (always evaluated on the geometric space).
  
|**<font color='red'>[FILL & RUN]</font>**|
|---|

In [ ]:
from ngsolve import Integrate

def J(u):
    """ integral of the half squared error """
    return Integrate( (u - uH1)**2  / 2 , Omega_h)

Since the problem is **strictly convex** and **infinite at infinite**, the minimum exists and is unique. Then, a good strategy to find $u^*$ is to cancel the derivative of $J$... *whatever that means*. Indeed, $u$ is not a scalar but a function.


|How can we derivate with respect to functions? |
|---|

#### 2.c.i) Directional derivative
Given any function $v\in H$ belonging to the same space than $u$, the directionnal derivative evaluates the first order variation of $J$ for a small step in the direction of $v$
$$J'(u; v) =  \lim_{t\rightarrow 0} \frac{J(u+tv) - J(u)}{t} $$

| **Exercise** (pen & paper)  : find the expression of $J'(u;v)$ |
|---|

#### 2.c.ii) Stationarity 
To solve the least square problem, it is enough to write the **optimality condition**, i.e, the derivative cancellation of $J$ :

$$ \forall v\in H(\Omega), \quad  J'(u;v) = 0$$
$$ \Rightarrow \boxed{\forall v\in H(\Omega),  \quad \underbrace{\int_\Omega u v}_{a(u,v)} = \underbrace{\int_\Omega u_{H^1} v}_{b(v)}} $$

which defines a linear system in $u$.

- The arbitrary direction $v$ is called a **test function**, from the chosen function space $H(\Omega)$
- The left-hand side $a(u,v)$ is a bilinear form
- The right hand side $b(v)$ is a linear form

#### 2.c.iii) Weak form

- The last line is called a **variational formulation**; it represents the **weak form** of the following "**strong**" equation, obtained by removing the integrals and test functions :
$$ u = u_{H^1}  $$
- The variational form is called **weak** because $u$ might not be exactly equal to $u_{H^1}$, it is only the *best possible* in the admissible space $H$.

`NGSolve` is a toolbox to solve variational formulations numerically. We can define linear and bilinear form, with the syntaxes :
- `a = BilinearForm( <symbolic expression> * dx)`
- or `a = BilinearForm(fes)`, and then `a += <symbolic expression> * dx` for longer expressions.
- `<symbolic expression> * dx("regionLabel")` is equivalent to restric the integral over a specific region.

The same syntaxic rules apply for `LinearForm`

|**<font color='red'>[FILL & RUN]</font>**|
|---|

In [ ]:
from ngsolve import dx, BilinearForm, LinearForm

# 1) Chose finite element space
fes = fesH1                   # should be fesL2 or fesH1

# 2) Define trial and test functions
u = fes.TrialFunction()       # DoFs
v = fes.TestFunction()        # test function

# 2) Define the forms
a = BilinearForm( u * v * dx )  # define the bilinear form
b = LinearForm( u_ref * v * dx )   # define the linear form

---
### 2.d) Assembly

Once discretized on a mesh and suitable function space with DoF indexed by $i$, we formally have:

- Bilinear form is equivalent to a matrix : $K_{ij} = a(u_j, v_i) $
- Linear form is equivalent to a vector $f_i = b(v_i)$

with $u_i, v_j$ the unitary function associated to DoF $i,j$, respectively. Building the matrix and the vector is called the "**assembly**" stage.

Then, one has to solve the matrix system :

$$ K_{ij} u_{j} = f_{i} $$

Efficient assembly is not a trivial task! It is related to computing pieces of the global integral on each elements. However, we don't have to worry about that: we just use the capabilities of `NGSolve`.

|**<font color='green'>[RUN & OBSERVE]</font>**|
|---|

In [ ]:
#Assembly step (difficult step, but NGsolve takes care of it!)
K = a.Assemble().mat
f = b.Assemble().vec

We can access the matrix and vizualize the sparsity pattern.

In [ ]:
import scipy.sparse as sp
import matplotlib.pylab as plt
rows, cols, vals = K.COO()
print(f"Non-zero terms ratio: {len(vals)/(K.height*K.width)*100:.2f} %")
plt.spy(sp.csr_matrix((vals,(rows,cols))))
plt.show()    # show non-zero terms of the matrix

---
### 2.e) Solving the problem and solution inspection

Solving the mean-square problem is now equivalent to solving this matrix system.

|**<font color='green'>[RUN & OBSERVE]</font>**|
|---|

In [ ]:
sol = GridFunction(fes)
sol.vec.data = K.Inverse() * f

# Visualize the solution
Draw(sol,  settings = {"deformation" :  0.5})

In [ ]:
# Print the LSE
print(f"{J(sol) = :.3e}")

Actually, least square error can be written as the square of the $L^2$ norm:
$$ ||u||_{L^2} =  \sqrt{\int_\Omega |u|^2 \; \mathrm{d}x} $$

In [ ]:
def L2norm(u):
    return Integrate(u**2, Omega_h) ** 0.5

print(f"L2norm(error) = {L2norm(sol - u_ref) :.3e}")


We can also define the $H^1$ norm, which makes sense only if $u\in H^1(\Omega)$:
$$ ||u||_{H^1} =  \sqrt{\int_\Omega |u|^2 + |\nabla u|^2\; \mathrm{d}x} $$


In [ ]:
from ngsolve import CoefficientFunction, grad as ngsgrad

def grad(u):
    try: return ngsgrad(u)
    except: return CoefficientFunction((u.Diff(x), u.Diff(y)))

def H1norm(u, grad_u):
    return L2norm(u) + L2norm(grad_u)

grad_u = grad(sol) - grad(u_ref)
print(f"H1norm(error) = {H1norm(sol-u_ref, grad_u) :.3e}")

## 3) Experimentation

Using the code above, **conduct experiments** aimed at reducing the error when approximating continuous and discontinuous functions. Identify the **error sources**. Play with:
- mesh size
- function space
- polynomial order

---
# Take home message
**The interpolation should be carefully chosen!**
 - **Not enough regularity** leads to ***slow convergence***, needing a lot of elements to decrease the approximation error enough
 - **Too much regularity** leads to ***wrong results***.

---
## To go further

The least mean square formulation minimize the square of $L^2$ norm. What if we want to minimize the square of the $H^1$ norm? Do we get the same result?

In [ ]:
a = BilinearForm(fes)
a += u * v * dx + grad(u) * grad(v) * dx
b = LinearForm( fes)
b += u_ref * v * dx  + grad(u_ref) * grad(v) * dx  # define the linear form

#Assembly step (difficult step, but NGsolve takes care of it!)
K = a.Assemble().mat
f = b.Assemble().vec

# solve
sol_ = GridFunction(fes)
sol_.vec.data = K.Inverse() * f

# Visualize the solution
Draw(sol_,  settings = {"deformation" :  0.5})

In [ ]:
# Print the LSE
print(f"{J(sol) = :.3e}")
print(f"L2norm(error) = {L2norm(sol_ - u_ref) :.3e}")
grad_u = grad(sol_) - grad(u_ref)
print(f"H1norm(error) = {H1norm(sol_-u_ref, grad_u) :.3e}")